# CottonLens V2 — kayıtları koruyarak devam
Önce takılan eski runtime'ı **Bağlantıyı kes ve çalışma zamanını sil** ile kapat. Başka bir notebook'ta bu deneyi çalıştırma.
T4 seç; ilk beş kod hücresini sırayla çalıştır. Dördüncü hücre eski kilit varsa yalnız eski runtime silindiğinde `SILDIM` yazmanı ister.
Kaynak V2 ve deney `research-v2-tf-placement` aynıdır. Tamamlanmış kayıtları engine doğrulayıp kullanır; kesilmiş fit yeniden çalışabilir.
Beşinci hücre Tur A'yı sürdürür. Bar yalnız kaydedilmiş fold kararlarını sayar; süre tahmini değildir. Aday 24/24 sonrası seed doğrulaması ve outer fit devam edebilir.
Gösterge yaklaşık üç saniyede bir CPU, son çıktı ve Drive log kopyası yaşını yeniler. On dakika çıktı yoksa kontrol uyarısı verir; bu tek başına kilitlenme kanıtı değildir.
Log yerel diske yazılır, ayrı iş parçacığıyla Drive'a kopyalanır. Drive gecikmesi göstergeyi durdurmaz; eğitim kodunun kendi Drive erişimi yine bekleyebilir.
Bu notebook araştırma sonucunu/ayarını değiştirmez ve otomatik release yayınlamaz.


In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/CottonLensAI')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
EXPERIMENT = 'research-v2-tf-placement'  # Retain this identifier for resumes of EXACTLY this data/source.


In [ ]:
import hashlib, importlib, json, sys, zipfile
from pathlib import PurePosixPath
SOURCE_ZIP = DRIVE_ROOT / 'sources/cottonlens-research-source-v2.zip'
expected_zip = SOURCE_ZIP.with_suffix('.zip.sha256').read_text().split()[0]
assert expected_zip == '8e134e2b72ae3d677076216590dac2dc745c3c400a1ac84d4c051533015f7a30', 'Expected frozen V2 ZIP'
assert hashlib.sha256(SOURCE_ZIP.read_bytes()).hexdigest() == expected_zip, 'Source ZIP checksum mismatch'
with zipfile.ZipFile(SOURCE_ZIP) as archive:
    source = json.loads(archive.read('.source-manifest.json'))
    payload = {key: value for key, value in source.items() if key != 'source_id'}
    identity = hashlib.sha256(json.dumps(payload, sort_keys=True, separators=(',', ':')).encode()).hexdigest()
    assert identity == source['source_id'], 'Source manifest mismatch'
    assert len(archive.namelist()) == len(set(archive.namelist())), 'Duplicate archive members'
    assert set(archive.namelist()) == set(source['files']) | {'.source-manifest.json'}, 'Unexpected archive files'
    REPO = Path('/content') / ('cottonlens-source-' + identity[:16])
    REPO.mkdir(exist_ok=True)
    for name in archive.namelist():
        member = PurePosixPath(name)
        assert not member.is_absolute() and '..' not in member.parts and chr(92) not in name and ':' not in name
        content = archive.read(name)
        if name in source['files']:
            assert hashlib.sha256(content).hexdigest() == source['files'][name], name
        target = REPO / name
        target.parent.mkdir(parents=True, exist_ok=True)
        if target.exists():
            assert target.read_bytes() == content, f'Existing Colab source changed: {name}'
        else:
            with target.open('xb') as handle:
                handle.write(content)
sys.path.insert(0, str(REPO / 'ml'))
import colab_setup
importlib.reload(colab_setup)
colab_setup.PROJECT = REPO
from colab_setup import setup, run, INFERENCE
print('Source identity:', identity, 'Git HEAD:', source['git_head'], 'Includes working-tree edits:', source['working_tree_dirty'])


In [ ]:
PYTHON = setup(research=True)  # Locked managed Python 3.12.11; no global packages.
print('Training interpreter:', PYTHON)
print('Inference-only interpreter:', INFERENCE / 'bin/python')


In [ ]:
import importlib.util
from IPython.display import display
helper_source = DRIVE_ROOT / 'sources/cottonlens-colab-progress-51940d329674903f.py'
helper_bytes = helper_source.read_bytes()
assert hashlib.sha256(helper_bytes).hexdigest() == '51940d329674903f3cfdb0d46b5b64c1d29713bc44e1563c6bbfd2d43c43cd11', 'Progress helper checksum mismatch'
helper_local = Path('/content/cottonlens-colab-progress-51940d329674903f.py')
helper_local.write_bytes(helper_bytes)
helper_spec = importlib.util.spec_from_file_location('cottonlens_progress', helper_local)
progress = importlib.util.module_from_spec(helper_spec)
helper_spec.loader.exec_module(progress)
experiment_root = DRIVE_ROOT / 'experiments' / EXPERIMENT
assert not progress.engine_processes(), 'A research process is still active on this runtime'
if (experiment_root / '.writer-lock').exists():
    answer = input('Eski runtime silindi ve bu deneyi çalıştıran başka oturum yoksa SILDIM yaz: ')
    progress.recover_empty_lock(experiment_root, confirmed_stopped=answer.strip() == 'SILDIM')
BASE = [PYTHON, '-m', 'cottonlens_ml.research.engine', '--repo', REPO, '--drive-root', DRIVE_ROOT, '--experiment', EXPERIMENT]
def supervised_stage(command):
    assert not progress.engine_processes(), 'A research process is still active on this runtime'
    status = display('Starting process monitor...', display_id=True)
    return progress.supervise(command, cwd=REPO, env=colab_setup.process_env(),
        drive_logs=DRIVE_ROOT / 'reports', decisions=experiment_root / 'decisions' / 'A',
        total=64, update=status.update)
prepare_result = supervised_stage([*BASE, '--stage', 'prepare'])


In [ ]:
ROUND = 'A'
EXTENSION = 0
round_result = supervised_stage([*BASE, '--stage', 'round', '--round', ROUND, '--extension', str(EXTENSION)])


## Aday kilitleme ve release
Arama turlarını değerlendirdikten sonra STAGE seç: lock → reproduce → export. Varsayılan compare hiçbir adayı kilitlemez. Kilitlenen deneyde yeni arama yapılmaz. Export yalnız yeniden üretim ve ayrı CPU runtime kontrolü geçerse ZIP oluşturur. Yeni sonuç eski demo veya DB üzerine kurulmaz.
CFTC/WASDE/kontrat verisi otomatik etkin değildir: gerçek yayın/vintage kanıtı içeren paket ayrı research kimliğiyle prepare aşamasına verilmelidir.

In [ ]:
STAGE = 'compare'  # Optional after round A completes; does not lock or export.
assert STAGE in ('compare', 'lock', 'reproduce', 'export')
release_result = supervised_stage([*BASE, '--stage', STAGE])
